# Systemy AI – Laboratorium 6
## Zadanie: klasyfikator recenzji IMDB na pełnym zbiorze

Zbuduj klasyfikator nastroju recenzji (pozytywna / negatywna) na **pełnym** zbiorze IMDB: 25 000 recenzji treningowych i 25 000 testowych. Wzoruj się na notatniku `SystemyAI_Lab6_IMDB_wzor.ipynb`; cały kod piszesz sam(a).

**Czas obliczeń.** Zbiór jest 5 razy większy niż we wzorze. Pojedyncza walidacja krzyżowa trwa kilkanaście–kilkadziesiąt sekund, strojenie w punkcie 4 – kilka minut. Dlatego używamy **3 rund** walidacji krzyżowej zamiast 5.

**Zasady** (jak poprzednio): nazwy zmiennych z treści są obowiązkowe; komórek `# TEST` nie zmieniaj; przed oddaniem *Uruchom ponownie i uruchom wszystko*.

---
## 1. Wczytanie i czyszczenie
1. Wczytaj zbiór `stanfordnlp/imdb` z Hugging Face do `train_df` i `test_df` (kolumny `text`, `label`). Skopiuj z wzoru wersję z zapasowym wczytaniem z GitHuba.
2. Zamień `<br />` na spacje w obu zbiorach.
3. Policz `class_share` (udział klas w `train_df`) i `median_words` (mediana liczby słów w recenzji treningowej).

In [4]:
# Twoje rozwiązanie
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
def wyczysc(df):
    df["text"] = df["text"].str.replace("<br />", " ")
    return df
train_df = wyczysc(train_df)
test_df = wyczysc(test_df)
class_share = train_df["label"].value_counts(normalize=True)
median_words = train_df["text"].apply(lambda x: len(x.split())).median()

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# TEST
assert train_df.shape == (25000, 2) and test_df.shape == (25000, 2)
assert abs(class_share[1] - 0.5) < 0.01
assert not train_df["text"].str.contains("<br />").any()
assert 150 < median_words < 200
print("1 OK")

1 OK


---
## 2. Porównanie trzech klasyfikatorów
1. Utwórz `cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)`.
2. Dla pipeline'ów `TfidfVectorizer()` + klasyfikator: `MultinomialNB()`, `LinearSVC()`, `LogisticRegression(max_iter=1000)` policz `cross_validate` na **całym** zbiorze treningowym.
3. Wyniki zbierz w tabeli `porownanie` (indeks: `"naiwny Bayes"`, `"liniowy SVM"`, `"regresja logistyczna"`; kolumny: `"CV"`, `"CV odch."`, `"czas uczenia [s]"`).

In [5]:
# Twoje rozwiązanie
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
porownanie = pd.DataFrame(columns=["CV", "CV odch.", "czas uczenia [s]"])
porownanie = porownanie.reindex(index=["naiwny Bayes", "liniowy SVM", "regresja logistyczna"])

modele = [
    make_pipeline(TfidfVectorizer(), MultinomialNB()),
    make_pipeline(TfidfVectorizer(), LinearSVC()),
    make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
]
nazwy = ["naiwny Bayes", "liniowy SVM", "regresja logistyczna"]

for model, nazwa in zip(modele, nazwy):
    wyniki = cross_validate(model, train_df["text"], train_df["label"], cv=cv, scoring="accuracy", return_train_score=False)

    porownanie.loc[nazwa, "CV"] = round(np.mean(wyniki["test_score"]), 4)
    porownanie.loc[nazwa, "CV odch."] = round(np.std(wyniki["test_score"]), 4)
    porownanie.loc[nazwa, "czas uczenia [s]"] = round(np.mean(wyniki["fit_time"]), 2)

    print(f"{nazwa}: {porownanie.loc[nazwa, 'CV']}")

naiwny Bayes: 0.8617
liniowy SVM: 0.8932
regresja logistyczna: 0.8854


In [ ]:
# TEST
assert list(porownanie.index) == ["naiwny Bayes", "liniowy SVM", "regresja logistyczna"]
assert porownanie["CV"].between(0.83, 0.92).all()
assert porownanie.loc["liniowy SVM", "CV"] > porownanie.loc["naiwny Bayes", "CV"]
print("2 OK")

2 OK


*Twoja odpowiedź:* porównaj wyniki z wynikami z wzoru (podzbiór 5000 recenzji). Co dało pięciokrotne zwiększenie zbioru treningowego? Czy kolejność modeli się zmieniła?

---
## 3. Strojenie: `GridSearchCV`
Dostrój pipeline `TfidfVectorizer(min_df=2)` + `LinearSVC()`:
- `tfidfvectorizer__ngram_range` ∈ {(1, 1), (1, 2)},
- `linearsvc__C` ∈ {0.1, 1}.

Obiekt zapisz w `gs` (z `cv=cv`), wypisz `best_params_` i `best_score_`. **To potrwa kilka minut** – w tym czasie odpowiedz na pytanie pod komórką.

In [6]:
# Twoje rozwiązanie
from sklearn.model_selection import GridSearchCV

param_grid = {
    "tfidfvectorizer__ngram_range": [(1, 1), (1, 2)],
    "linearsvc__C": [0.1, 1]
}
gs = GridSearchCV(make_pipeline(TfidfVectorizer(min_df=2), LinearSVC()), param_grid, cv=cv, scoring="accuracy")
gs.fit(train_df["text"], train_df["label"])
print(gs.best_params_)
print(gs.best_score_)

{'linearsvc__C': 1, 'tfidfvectorizer__ngram_range': (1, 2)}
0.9064398669059314


In [ ]:
# TEST
assert gs.best_params_["tfidfvectorizer__ngram_range"] == (1, 2), "Na pełnym zbiorze bigramy powinny pomagać"
assert gs.best_score_ > porownanie.loc["liniowy SVM", "CV"]
print("3 OK")

3 OK


*Twoja odpowiedź:* we wzorze bigramy na 5000 recenzjach nic nie dały. Czy na pełnym zbiorze jest inaczej? Dlaczego większy zbiór pomaga właśnie bigramom? Ile razy wolniej uczy się model z bigramami i dlaczego?

---
## 4. Ocena końcowa
`model_koncowy = gs.best_estimator_`. Policz `acc_test` (accuracy na **całym** zbiorze testowym – jedyny raz, kiedy go używasz), narysuj macierz pomyłek i wypisz `classification_report`.

In [7]:
# Twoje rozwiązanie
model_koncowy = gs.best_estimator_
y_test = test_df["label"]
y_pred = model_koncowy.predict(test_df["text"])
acc_test = (y_pred == y_test).mean()
print(acc_test)

0.90296


In [ ]:
# TEST
assert 0.87 < acc_test < 0.93
print("4 OK")

4 OK


---
## 5. Co model „wie” i gdzie się myli
1. Z `model_koncowy` wyciągnij nazwy cech (`model_koncowy[0].get_feature_names_out()`) i wagi (`model_koncowy[-1].coef_[0]`) do `pd.Series` o nazwie `wagi`. Wypisz po 15 cech o największej i najmniejszej wadze. Czy są wśród nich bigramy?
2. `bledy` – `DataFrame` z recenzjami testowymi, które model sklasyfikował źle (kolumny `text`, `label`, `pred`). Wypisz liczbę błędów i przeczytaj **3 przykłady** (pierwsze 500 znaków).
3. Napisz **5 własnych recenzji** (po angielsku), w tym co najmniej jedną z negacją i jedną ironiczną, i sprawdź predykcje modelu.

In [8]:
# Twoje rozwiązanie
wagi = pd.Series(model_koncowy[1].coef_[0], index=model_koncowy[0].get_feature_names_out()).sort_values()
print(wagi.head(15))
print(wagi.tail(15))
bledy = test_df.copy()
bledy["pred"] = y_pred
bledy = bledy[bledy["label"] != bledy["pred"]]
print(bledy.shape[0])
print(bledy["text"].head(500))


worst            -4.186860
awful            -3.717987
bad              -3.639967
boring           -3.461435
poor             -3.270281
the worst        -3.183395
waste            -2.919468
disappointment   -2.604945
terrible         -2.583010
worse            -2.566881
poorly           -2.510668
disappointing    -2.462955
dull             -2.420053
unfortunately    -2.395830
annoying         -2.333171
dtype: float64
best         1.835280
favorite     1.853958
bit          1.855965
loved        1.876260
superb       1.906072
brilliant    1.945156
the best     1.955429
enjoyable    1.964973
today        2.203575
amazing      2.208377
fun          2.286092
wonderful    2.565872
perfect      2.751831
excellent    3.236597
great        3.571217
dtype: float64
2426
4       First off let me say, If you haven't enjoyed a...
18      Ben, (Rupert Grint), is a deeply unhappy adole...
32      I'm the type of guy who loves hood movies from...
79      It's Saturday, it's raining, and I think every..

In [9]:
# TEST
assert len(wagi) > 100_000, "Słownik powinien zawierać bigramy"
assert set(bledy.columns) >= {"text", "label", "pred"} and len(bledy) == (y_pred != y_test.values).sum()
assert any(" " in w for w in wagi.head(50).index) or any(" " in w for w in wagi.tail(50).index)
print("5 OK")

5 OK


In [ ]:
print(bledy['text'].head(3))

ID: 4 | Prawdziwa etykieta: 0 | Predykcja: 1

First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy these kinds of movies for what they are. This movie is much better than any of the movies the other action guys (Segal and Dolph) have thought about putting out the past few years. Van Damme is good in the movie, the movie is only worth watching to Van Damme fans. It is not as good as Wake of Death (which i highly recommend to anyone of likes Van Damme) or In hell but, in my opinion it's worth watching. It has the same type of feel to it as Nowhere to Run. Good fun stuff!


ID: 18 | Prawdziwa etykieta: 0 | Predykcja: 1

Ben, (Rupert Grint), is a deeply unhappy adolescent, the son of his unhappily married parents. His father, (Nicholas Farrell), is a vicar and his mother, (Laura Linney), is ... well, let's just say she's a somewhat hypocritical sold

In [10]:
# Wyświetlenie wyłącznie bigramów wśród cech o największej i najmniejszej wadze
bigramy_wagi = wagi[wagi.index.str.contains(' ')]
print("Najważniejsze negatywne bigramy:")
print(bigramy_wagi.head(10))
print("\nNajważniejsze pozytywne bigramy:")
print(bigramy_wagi.tail(10))

Najważniejsze negatywne bigramy:
the worst      -3.183395
not worth      -1.926074
at all         -1.866331
waste of       -1.691102
than this      -1.680275
not good       -1.615698
at best        -1.537989
not even       -1.533275
not very       -1.443984
the original   -1.436952
dtype: float64

Najważniejsze pozytywne bigramy:
on dvd              1.344837
watch it            1.356774
very well           1.362169
very good           1.379135
definitely worth    1.435620
well worth          1.540778
must see            1.590395
this is             1.636565
better than         1.675651
the best            1.955429
dtype: float64


In [11]:
# 5 własnych recenzji do przetestowania (0 = negatywna, 1 = pozytywna)
moje_recenzje = [
    "This movie was absolutely brilliant, the acting was top notch and the plot kept me on the edge of my seat.",  # Zwykła pozytywna
    "I fell asleep after 10 minutes. It was extremely boring and a total waste of money.",  # Zwykła negatywna
    "This was not a bad movie at all, in fact, I actually quite enjoyed it despite the negative reviews.",  # Negacja
    "Oh, what a masterpiece! I especially loved the part where nothing happened for two hours.",  # Ironia / Sarkazm
    "I wanted to hate this film, but the incredible performance by the main cast made it impossible."  # Złożona / Przewrotna
]

prawdziwe_etykiety = [1, 0, 1, 0, 1]
predykcje = model_koncowy.predict(moje_recenzje)

for i, (recenzja, true_lbl, pred_lbl) in enumerate(zip(moje_recenzje, prawdziwe_etykiety, predykcje)):
    print(f"Recenzja {i+1}: {recenzja[:60]}...")
    print(f"  Prawdziwa: {true_lbl} | Predykcja: {pred_lbl} -> {'OK' if true_lbl == pred_lbl else 'BŁĄD'}\n")

Recenzja 1: This movie was absolutely brilliant, the acting was top notc...
  Prawdziwa: 1 | Predykcja: 1 -> OK

Recenzja 2: I fell asleep after 10 minutes. It was extremely boring and ...
  Prawdziwa: 0 | Predykcja: 0 -> OK

Recenzja 3: This was not a bad movie at all, in fact, I actually quite e...
  Prawdziwa: 1 | Predykcja: 1 -> OK

Recenzja 4: Oh, what a masterpiece! I especially loved the part where no...
  Prawdziwa: 0 | Predykcja: 1 -> BŁĄD

Recenzja 5: I wanted to hate this film, but the incredible performance b...
  Prawdziwa: 1 | Predykcja: 1 -> OK



*Twoja odpowiedź:*
1. Jakie bigramy znalazły się wśród najważniejszych cech? Co mówią o tym, czego model się nauczył?

wśród cech najważniejsze biagramy to "the worst" dla negatywnych oraz "on dvd" dla pozytywnych warto zauważyć zę biagram negatywny ma dużo wyższą wartość co oznacza że model skorelował go mocno wystąpienie zwrotu z negatywnym wydźwiękiem recenzji
2. Opisz trzy przeczytane błędy: dlaczego model mógł się pomylić? Czy człowiek też miałby wątpliwości?

ID 4 (recenzja Van Damme): w recenzji całościowo negatywnej autor zawarł zwroty brzmiące pozytywnie przez co całościowo model uznał ją za pozytywną

ID 18 (recenzja filmu z Rupert Grint): w negatywnej recenzji autor pozytywnie opisał aktorów

ID 32 (recenzja filmu klasy B): jest to recenzja negatywna która unika jednoznacznie negatywnych określeń. w przypadku człowieka uwagę zwróciłby zwrot Don't WATCH IT nakierowując czytelnika o tym że recenzja jest negatywna

3. Które z Twoich recenzji model ocenił źle? Co mają wspólnego?

model ocenił recenzję zawierającą sarkazm błędnie co jest zrozumiałe ze względu na naturę sarkazmu.

4. Gdzie leży granica podejścia „worek słów + model liniowy”? Co musiałby umieć model, żeby poradzić sobie z Twoimi trudnymi przykładami?

Granica podejścia TF-IDF + model liniowy leży w braku zrozumienia kontekstu, szyku zdania oraz semantyki.